# Data wrangling and landing labels

The 90-row course sample excludes Falcon 1 and multiple-payload records. Known payload values are numeric; the frozen file already mean-imputes missing mass. For modeling we reverse its exact mean sentinel and fit median imputation inside each training fold.

Class 1 includes True Ocean. Class 0 groups failure, no attempt and unknown outcomes. This target does not prove recovery for reuse.

In [1]:
"""Reproducible SpaceX capstone: scraping, SQL, visual EDA and model evaluation."""
from pathlib import Path
from io import StringIO
import calendar
import json
import re
import sqlite3
import warnings

import numpy as np
import pandas as pd
import matplotlib

import matplotlib.pyplot as plt
import seaborn as sns
from bs4 import BeautifulSoup
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.model_selection import GridSearchCV, StratifiedKFold, TimeSeriesSplit, train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.tree import DecisionTreeClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import accuracy_score, balanced_accuracy_score, f1_score, precision_score, recall_score, confusion_matrix


ROOT = Path.cwd().parent
DATA = ROOT / 'data'
FIG = ROOT / 'figures'


In [2]:
df = pd.read_csv(DATA/'dataset_part_1.csv')
display(df.isna().sum().to_frame('Missing'))
display(df.Outcome.value_counts().to_frame('Launches'))
bad_outcomes = {'False ASDS','False Ocean','False RTLS','None ASDS','None None'}
df['Class'] = df.Outcome.map(lambda outcome: int(outcome not in bad_outcomes))
reference = pd.read_csv(DATA/'dataset_part_2.csv')
assert df.Class.equals(reference.Class)
assert df.FlightNumber.is_unique
print('Successes:',df.Class.sum(), 'of',len(df))
display(df.groupby('LaunchSite').Class.agg(['sum','size','mean']))
df.to_csv(DATA/'wrangled_dataset.csv',index=False)

,Missing
FlightNumber,0
Date,0
BoosterVersion,0
PayloadMass,0
Orbit,0
LaunchSite,0
Outcome,0
Flights,0
GridFins,0
Reused,0


,Launches
Outcome,
True ASDS,41
None None,19
True RTLS,14
False ASDS,6
True Ocean,5
False Ocean,2
None ASDS,2
False RTLS,1


Successes: 60 of 90


,sum,size,mean
LaunchSite,,,
CCAFS SLC 40,33,55,0.600000
KSC LC 39A,17,22,0.772727
VAFB SLC 4E,10,13,0.769231


Prelaunch feature discipline: omit Outcome, Class, LandingPad, Serial and eventual ReusedCount as predictors. FlightNumber is an era proxy; temporal validation tests this concern. Coordinates are used for mapping.